In [1]:
import os
import pandas
import shutil

# Ensure the directory exists and is accessible
working_dir = '/kaggle/working/BoostTrack'

# Check if the directory already exists and remove it if it does
if os.path.isdir(working_dir):
    shutil.rmtree(working_dir)

# Make sure the working directory is valid and accessible
os.makedirs(working_dir, exist_ok=True)

# Change to the working directory before cloning the repo
os.chdir('/kaggle/working')

# Clone the repository again
# !git clone -b arguments https://github.com/DomaMorcos/CustomBoostTrack BoostTrack
!git clone  https://github.com/DomaMorcos/CustomBoostTrack.git BoostTrack

os.chdir("BoostTrack")
os.getcwd()

Cloning into 'BoostTrack'...
remote: Enumerating objects: 1325, done.
remote: Counting objects: 100% (1325/1325), done.
remote: Compressing objects: 100% (992/992), done.
remote: Total 1325 (delta 298), reused 1277 (delta 251), pack-reused 0 (from 0)
Receiving objects: 100% (1325/1325), 18.76 MiB | 21.32 MiB/s, done.
Resolving deltas: 100% (298/298), done.


'/kaggle/working/BoostTrack'

In [2]:
!pip install 'git+https://github.com/ifzhang/ByteTrack.git' -q
!pip install ultralytics
!pip install torchreid
!pip install lap
!pip install yacs
!pip install loguru
!pip install ensemble_boxes
!pip install thop

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 71.7 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.8.93
    Uninstalling nvidia-nvjitlink-cu12-12.8.93:
      Successfully uninstalled nvidia-nvjitlink-cu12-12.8.93
  Attempting uninstall: nvidia-curand-cu12
    Found existing installation: nvidia-curand-cu12 10.3.9.90


In [3]:
import cv2
import os

def extract_frames(video_path, output_folder, speed = 3): # speed like 2x, 3x, 4x
    """
    Extract frames from a video at 3x speed (keeping only 1/3 of the frames) 
    and save them as images in a folder.
    
    Args:
        video_path (str): Path to the input video file
        output_folder (str): Path to the folder where frames will be saved
    """
    # Create the output folder if it doesn't exist
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    
    # Open the video file
    cap = cv2.VideoCapture(video_path)
    
    if not cap.isOpened():
        print("Error: Could not open video file")
        return
    
    # Get video properties
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    print(f"Video Info:")
    print(f" - Name: {os.path.basename(video_path)}")
    print(f" - Frames: {frame_count}")
    print(f" - FPS: {fps}")
    print(f" - Resolution: {width}x{height}")
    
    # Read and save every 3rd frame (for 3x speed)
    saved_frame_number = 0
    original_frame_number = 0
    
    while True:
        ret, frame = cap.read()
        
        if not ret:
            break
        
        # Only save every 3rd frame (0, 3, 6, etc.)
        if original_frame_number % speed == 0:
            # Save frame as an image file
            frame_filename = os.path.join(output_folder, f"frame_{saved_frame_number:06d}.jpg")
            cv2.imwrite(frame_filename, frame)
            saved_frame_number += 1
            
            # Print progress every 100 saved frames
            if saved_frame_number % 100 == 0:
                print(f"Processed frame {original_frame_number}/{frame_count} (saved {saved_frame_number})")
        
        original_frame_number += 1

    
    # Release resources
    cap.release()
    print(f"\nFinished! Saved {saved_frame_number} frames (1/{speed} of original) to {output_folder}")

In [4]:
import pandas as pd
import cv2
import os
import random

def draw_tracking_results_on_frames(frames_folder_path, annnots_path, annoted_video_path, frame_rate):
    images = [os.path.join(frames_folder_path, path) for path in os.listdir(frames_folder_path)]
    images = sorted(images)
    gt_df = pd.read_csv(annnots_path, header=None)
    gt_df.columns = ['frame', 'track_id', 'x', 'y', 'width', 'height', 'confidence', 'x_', 'y_', 'z_']

    first_image = cv2.imread(images[0])
    height, width, _ = first_image.shape
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    video_writer = cv2.VideoWriter(annoted_video_path, fourcc, frame_rate, (width, height))
    
    # Dictionary to store colors for each track ID
    id_to_color = {}

    for frame_id in range(1, len(images)+1):
    # for frame_id in gt_df['frame'].unique():
        frame_path = images[frame_id-1]
        img = cv2.imread(frame_path)

        # Get the detections for this frame
        detections = gt_df[gt_df['frame'] == frame_id]

        # Draw each detection on the image
        for _, row in detections.iterrows():
            track_id = int(row['track_id'])
            x = int(row['x'])
            y = int(row['y'])
            width = int(row['width'])
            height = int(row['height'])
            confidence = row['confidence']

            # Generate a random color for each track ID
            if track_id not in id_to_color:
                id_to_color[track_id] = [random.randint(0, 255) for _ in range(3)]

            color = tuple(id_to_color[track_id])
            cv2.rectangle(img, (x, y), (x + width, y + height), color, 2)
            cv2.putText(img, f"ID: {track_id}", (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 1, color, 3)

        video_writer.write(img)
    video_writer.release()
    print("video is Done")

In [5]:
import os, cv2, shutil
from tqdm import tqdm
cwd = "/kaggle/working/"
main_path = "/kaggle/input/pmfawrydata"
video_paths = [os.path.join(main_path, path) for path in os.listdir(main_path)]
video_paths = sorted(video_paths)
video_paths = [video_paths[4]]

for video_path in tqdm(video_paths, desc = "Processing videos"):
    frames_path  = os.path.join(cwd, os.path.basename(video_path).split('.')[0])
    output_video = os.path.join(cwd, os.path.basename(video_path))
    os.makedirs(frames_path, exist_ok=True)
    extract_frames(video_path, frames_path, speed=1)
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise ValueError("Could not open the video file")
        
    fps = cap.get(cv2.CAP_PROP_FPS)
    
    dataset_name = "tarsh"
    output_folder = os.path.join(cwd, "results")
    dataset_path = frames_path
    reid_path = "/kaggle/input/finalosnetbest/OSNet_Best.pth"
    frame_rate = int(fps)
    model1_path = "/kaggle/input/yolo11-last/yolo11x-mot-finetune/checkpoints/yolo11x-final/weights/best.pt"
    model1_weight = 0.7
    model2_path = "/kaggle/input/yolo12last/yolo12x-mot-finetune/checkpoints/yolo12x-final/weights/best.pt"
    model2_weight = 0.3
    print(frame_rate)

    !python run_with_yolo.py --dataset {dataset_name} --exp_name BTPP --result_folder {output_folder} \
    --frame_rate {frame_rate} --reid_path {reid_path} --dataset_path {dataset_path} \
    --yolo_path {model1_path} 
    # !python run_with_ensembler.py --dataset {dataset_name} --exp_name BTPP --result_folder {output_folder} \
    # --frame_rate {frame_rate} --reid_path {reid_path} --dataset_path {dataset_path} \
    # --model1_path {model1_path} --model1_weight {model1_weight} \
    # --model2_path {model2_path} --model2_weight {model2_weight} \
    # --conf_thresh 0.1

    speed_of_output_video = 2
    draw_tracking_results_on_frames(frames_path, os.path.join(output_folder, "BTPP_post_gbi/data/test.txt"), output_video, fps*speed_of_output_video)
    shutil.rmtree(frames_path)
    shutil.rmtree(output_folder)

Processing videos:   0%|          | 0/1 [00:00<?, ?it/s]

Video Info:
 - Name: D7_S20250412210000_E20250412210536.mp4
 - Frames: 8434
 - FPS: 25.0
 - Resolution: 1920x1080
Processed frame 99/8434 (saved 100)
Processed frame 199/8434 (saved 200)
Processed frame 299/8434 (saved 300)
Processed frame 399/8434 (saved 400)
Processed frame 499/8434 (saved 500)
Processed frame 599/8434 (saved 600)
Processed frame 699/8434 (saved 700)
Processed frame 799/8434 (saved 800)
Processed frame 899/8434 (saved 900)
Processed frame 999/8434 (saved 1000)
Processed frame 1099/8434 (saved 1100)
Processed frame 1199/8434 (saved 1200)
Processed frame 1299/8434 (saved 1300)
Processed frame 1399/8434 (saved 1400)
Processed frame 1499/8434 (saved 1500)
Processed frame 1599/8434 (saved 1600)
Processed frame 1699/8434 (saved 1700)
Processed frame 1799/8434 (saved 1800)
Processed frame 1899/8434 (saved 1900)
Processed frame 1999/8434 (saved 2000)
Processed frame 2099/8434 (saved 2100)
Processed frame 2199/8434 (saved 2200)
Processed frame 2299/8434 (saved 2300)
Processed

Processing videos: 100%|██████████| 1/1 [45:38<00:00, 2739.00s/it]


In [6]:
    # speed_of_output_video = 2
    # draw_tracking_results_on_frames(frames_path,"/kaggle/working/results/tarsh-val/BTPP_post_gbi/data/test.txt", output_video, fps*speed_of_output_video)
    # shutil.rmtree(frames_path)
    # shutil.rmtree(output_folder)

In [7]:
import shutil
shutil.rmtree("/kaggle/working/BoostTrack")